# Phase 5: Legal Statute Identification: Strided GQA decoder

This is the **Legal Statute Identification** part of the full `strided` notebook (`notebooks/CS781_strided_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Legal Statute Identification

In [15]:
import json, os, pickle, time, numpy as np
from collections import Counter
from huggingface_hub import snapshot_download

P5_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_5"
os.makedirs(P5_DIR, exist_ok=True)

local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_5_LSI/*", "*/Phase_5_LSI/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_5" in r: return os.path.join(r, name)
tr, dv, te = [json.load(open(find(n))) for n in ("train.json", "dev.json", "test.json")]
print("sizes:", len(tr), len(dv), len(te))
print("train keys:", list(tr[0].keys()), "| test keys:", list(te[0].keys()))
print("example labels:", tr[0]["labels"][:10], "| text start:", tr[0]["text"][:100].replace("\n", " "))

# ---- label statistics ----
ctr, ctd = Counter(l for r in tr for l in r["labels"]), Counter(l for r in dv for l in r["labels"])
allids = sorted(set(ctr) | set(ctd))
print(f"\nstatute ids: train {len(ctr)} | dev {len(ctd)} | union {len(allids)} | min id {min(allids)} max id {max(allids)}")
print("dev ids unseen in train:", len(set(ctd) - set(ctr)), "| train ids absent in dev:", len(set(ctr) - set(ctd)))
fr = np.array(sorted(ctr.values(), reverse=True))
print("train positives per statute: max", fr[0], "median", int(np.median(fr)), "min", fr[-1])
print("statutes with <=5 train positives:", int((fr <= 5).sum()), "| <=20:", int((fr <= 20).sum()), "| >=500:", int((fr >= 500).sum()))
nl = np.array([len(r["labels"]) for r in tr])
print("labels per train doc: mean %.2f median %d max %d | docs with 1 label: %.1f%%" % (nl.mean(), np.median(nl), nl.max(), 100 * (nl == 1).mean()))
print("duplicate ids inside a doc:", sum(len(r["labels"]) != len(set(r["labels"])) for r in tr))
print("top 10 statutes (id, train count):", ctr.most_common(10))

# ---- tokenise once: keep first/last 4096 tokens (same scheme as Phase 4) ----
def tok_split(texts, N=4096):
    out = []
    for i in range(0, len(texts), 500):
        b = texts[i:i + 500]
        eh = tokenizer.encode_batch([t[:40000] for t in b])
        long_ = [j for j, t in enumerate(b) if len(t) > 80000]
        et = dict(zip(long_, tokenizer.encode_batch([b[j][-40000:] for j in long_]))) if long_ else {}
        for j in range(len(b)):
            hi = eh[j].ids; ti, total = (et[j].ids, 10**6) if j in et else (hi, len(hi))
            out.append((np.array(hi[:N], dtype=np.uint16), np.array(ti[-N:], dtype=np.uint16), total))
    return out

CACHE = f"{P5_DIR}/lsi_tok_cache.pkl"
if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        tk = tok_split([r["text"] for r in d])
        C[name] = dict(ids=[r["id"] for r in d], labels=[r.get("labels") for r in d],
                       head=[a for a, _, _ in tk], tail=[b for _, b, _ in tk], n=np.array([n for _, _, n in tk]))
        print(name, f"{(time.time() - t0) / 60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))
n = C["train"]["n"]
print("tokens per train doc: median", int(np.median(n)), "| p90", int(np.percentile(n, 90)),
      "| <=4096: %.1f%%" % (100 * (n <= 4096).mean()), "| <=8192: %.1f%%" % (100 * (n <= 8192).mean()))
print("✅ Phase-5 cache ready")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

sizes: 42750 10181 13019
train keys: ['id', 'text', 'labels'] | test keys: ['id', 'text']
example labels: [69, 9, 3] | text start: (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this p

statute ids: train 100 | dev 100 | union 100 | min id 0 max id 99
dev ids unseen in train: 0 | train ids absent in dev: 0
train positives per statute: max 11731 median 679 min 272
statutes with <=5 train positives: 0 | <=20: 0 | >=500: 70
labels per train doc: mean 3.71 median 3 max 31 | docs with 1 label: 21.4%
duplicate ids inside a doc: 3
top 10 statutes (id, train count): [(5, 11731), (34, 10380), (97, 7881), (43, 7728), (39, 7121), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
loaded cache
tokens per train doc: median 663 | p90 4286 | <=4096: 89.2% | <=8192: 97.3%
✅ Phase-5 cache ready


In [16]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
assert "C" in globals() and "P5_DIR" in globals() and "RRModel" in globals(), "run the cache cell and the Phase-3 toolkit first"

N_LAB, CH = 100, 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}
GRID = np.arange(0.05, 0.80, 0.025)

def multi_hot(lists):
    Y = np.zeros((len(lists), N_LAB), np.float32)
    for i, l in enumerate(lists): Y[i, list(set(l))] = 1          # also removes duplicate ids
    return Y
YTR, YDV = multi_hot(C["train"]["labels"]), multi_hot(C["dev"]["labels"])

# ---------- document -> chunks (views over the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):
        ends = list(range(L, 0, -CH))[::-1]; return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

_CNT = {}
def chunk_counts(split, mode):
    if (split, mode) not in _CNT:
        _CNT[(split, mode)] = np.array([max(1, len(get_chunks(split, i, mode))) for i in range(len(C[split]["ids"]))])
    return _CNT[(split, mode)]

def make_batches(split, idxs, mode, max_chunks=16, rng=None, mega=64):
    """Batches of <= max_chunks chunks; documents of similar length are grouped to avoid padding waste."""
    cnt = chunk_counts(split, mode); n = np.minimum(C[split]["n"], 4*CH); idxs = np.asarray(idxs)
    if rng is not None:
        idxs = idxs[rng.permutation(len(idxs))]
        groups = [idxs[s:s+max_chunks*mega] for s in range(0, len(idxs), max_chunks*mega)]
        groups = [g[np.argsort(n[g], kind="stable")] for g in groups]
    else:
        groups = [idxs[np.argsort(n[idxs], kind="stable")]]
    out = []
    for g in groups:
        cur, c = [], 0
        for i in g:
            if cur and c + cnt[i] > max_chunks: out.append(np.array(cur)); cur, c = [], 0
            cur.append(i); c += cnt[i]
        if cur: out.append(np.array(cur))
    if rng is not None: out = [out[j] for j in rng.permutation(len(out))]
    return out

def collate_ls(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> per-chunk statute logits -> document logits ----------
class LSIModel(RRModel):
    agg = "mean"                                   # "mean" or "max" over chunks
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        lg = self.head(self.drop(torch.cat([last, mean], -1))).float()          # (chunks, N_LAB)
        if self.agg == "max":
            return torch.full((n_docs, lg.size(1)), -1e4, device=lg.device).scatter_reduce(
                0, owner[:, None].expand_as(lg), lg, "amax", include_self=False)
        D = torch.zeros(n_docs, lg.size(1), device=lg.device).index_add_(0, owner, lg)
        return D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]

def ls_from(rr):
    """Phase-3 RRModel -> LSI model: same LM, fresh 100-way head (bias = log prior odds of each statute)."""
    d = rr.lm.token_embedding.weight.shape[1]
    m = LSIModel(rr.lm, d, N_LAB, 0.1).to(device)
    m.head = nn.Linear(2 * d, N_LAB).to(device)
    nn.init.normal_(m.head.weight, std=0.02)
    p = YTR.mean(0).clip(1e-4, 1 - 1e-4)
    with torch.no_grad(): m.head.bias.copy_(torch.tensor(np.log(p / (1 - p)), dtype=torch.float))
    return m

# ---------- inference, decoding, metrics ----------
@torch.no_grad()
def ls_logits(model, split, mode, idxs=None, max_chunks=32):
    model.eval(); N = len(C[split]["ids"]); idxs = np.arange(N) if idxs is None else np.asarray(idxs)
    out = np.zeros((N, N_LAB), np.float32)
    for b in make_batches(split, idxs, mode, max_chunks, rng=None):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out[b] = model(*collate_ls(split, b, mode)).float().cpu().numpy()
    return out[idxs]

def sigmoid(z): return 1 / (1 + np.exp(-z))

def decode(P, thr, force_one=True):
    pred = P >= thr
    if force_one:
        none = ~pred.any(1); pred[none, P[none].argmax(1)] = True      # every document has at least one statute
    return pred

def ls_scores(Y, pred):
    Yb = Y.astype(bool); tp = (pred & Yb).sum(0); fp = (pred & ~Yb).sum(0); fn = (~pred & Yb).sum(0)
    f = np.where(2*tp + fp + fn > 0, 2*tp / np.maximum(2*tp + fp + fn, 1), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0); r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return dict(macro=float(f.mean()), p=float(p.mean()), r=float(r.mean()),
                micro=float(2*tp.sum() / max(2*tp.sum() + fp.sum() + fn.sum(), 1)), subset=float((pred == Yb).all(1).mean()), per_label=f)

def best_global(Y, P):
    best = (0.5, -1.0)
    for t in GRID:
        f = ls_scores(Y, decode(P, t))["macro"]
        if f > best[1]: best = (float(t), f)
    return best

def best_per_label(Y, P):
    Yb = Y.astype(bool); thr = np.full(N_LAB, 0.5)
    for j in range(N_LAB):
        pj = P[:, j][:, None] >= GRID[None, :]; yj = Yb[:, j][:, None]
        tp = (pj & yj).sum(0); fp = (pj & ~yj).sum(0); fn = (~pj & yj).sum(0)
        thr[j] = GRID[(2*tp / np.maximum(2*tp + fp + fn, 1)).argmax()]
    return thr

def honest_perlabel(Y, P):
    """Per-label thresholds tuned on one half of dev and scored on the other half (and vice versa)."""
    idx = np.arange(len(Y)); A, B = idx[::2], idx[1::2]; out = []
    for a, b in [(A, B), (B, A)]:
        out.append(ls_scores(Y[b], decode(P[b], best_per_label(Y[a], P[a])))["macro"])
    return float(np.mean(out))

# ---------- training (BCE / weighted BCE, optional KD from cached teacher logits) ----------
def train_ls(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, max_chunks=16, n_train=None,
             loss_type="wbce", kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed); ntr = len(YTR)
    dev_idx = np.arange(300) if debug else np.arange(len(YDV)); Ydev = YDV[dev_idx]
    tr_idx = rng.permutation(ntr)[: (160 if debug else (n_train or ntr))]
    if debug: epochs, evals_per_epoch = 1, 1
    pw = None
    if loss_type == "wbce":
        pos = YTR.sum(0); pw = torch.tensor(np.clip(np.sqrt((ntr - pos) / pos), 1, 10), dtype=torch.float, device=device)
    ytr_t = torch.tensor(YTR); kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None
    head_p = list(model.head.parameters()); hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    spe = len(make_batches("train", tr_idx, mode, max_chunks, np.random.RandomState(seed))); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} loss={loss_type} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P5_DIR}/ls_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time(); done = False
    torch.cuda.reset_peak_memory_stats()
    for ep in range(epochs):
        model.train()
        for b in make_batches("train", tr_idx, mode, max_chunks, rng):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_ls("train", b, mode)).float()
            y = ytr_t[b].to(device)
            loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kdl = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(t_lg / T)) * T * T
                loss = (1 - alpha) * loss + alpha * kdl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 200 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P); model.train()
                s5 = ls_scores(Ydev, decode(P, 0.5))
                print(f"  eval@{step}: macro-F1 @0.5 {s5['macro']:.4f} | @best global thr {thr:.3f}: {f:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if f > best[0]:
                    best = (f, step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path): model.load_state_dict(torch.load(path), strict=False)
    P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P)
    sg = ls_scores(Ydev, decode(P, thr)); tl = best_per_label(Ydev, P); sl = ls_scores(Ydev, decode(P, tl))
    hon = honest_perlabel(Ydev, P)
    res = dict(name=name, mode=mode, thr_global=thr, thr_label=tl, f1_at_05=ls_scores(Ydev, decode(P, 0.5))["macro"],
               macro_global=sg["macro"], p=sg["p"], r=sg["r"], micro=sg["micro"], subset=sg["subset"],
               macro_perlabel_optimistic=sl["macro"], macro_perlabel_honest=hon, per_label=sg["per_label"],
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9, trainable=n_tr, total=n_all, dev_probs=P)
    print(f"✅ [{name}] dev macro-F1: @0.5 {res['f1_at_05']:.4f} | global thr {thr:.3f} → {sg['macro']:.4f} "
          f"(P {sg['p']:.4f} R {sg['r']:.4f}, micro {sg['micro']:.4f}, subset acc {sg['subset']:.4f})\n"
          f"   per-label thresholds: {sl['macro']:.4f} on the tuning data, {hon:.4f} on held-out dev halves "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB")
    return res

# ---------- outputs ----------
def write_pred_ls(model, name, mode, res):
    L = ls_logits(model, "test", mode); P = sigmoid(L)
    np.save(f"{P5_DIR}/test_probs_{name}.npy", P); np.save(f"{P5_DIR}/dev_probs_{name}.npy", res["dev_probs"])
    for tag, thr in [("global", res["thr_global"]), ("perlabel", res["thr_label"])]:
        pred = decode(P, thr); ids = C["test"]["ids"]
        out = [{"id": i, "pred": [int(k) for k in np.flatnonzero(pr)]} for i, pr in zip(ids, pred)]
        assert len(out) == len(ids) == 13019 and all(0 < len(o["pred"]) == len(set(o["pred"])) and max(o["pred"]) < N_LAB for o in out)
        f = f"{P5_DIR}/prediction_{name}_{tag}.json"; json.dump(out, open(f, "w"))
        print(f"wrote {f} | mean statutes/doc {pred.sum(1).mean():.2f}")

def save_kd_ls(model, name, mode):
    L = ls_logits(model, "train", mode); np.save(f"{P5_DIR}/kd_logits_{name}.npy", L); return L

print("✅ Phase-5 toolkit ready")

✅ Phase-5 toolkit ready


In [ ]:
SECOND_CKPT = f"{DRIVE_PROJECT_DIR}/alt_stride_32_128_checkpoints/best.pt"
assert os.path.exists(SECOND_CKPT), SECOND_CKPT
for n in ["CustomDecoderLM", "alternating_patterns", "ls_from", "train_ls", "collate_ls", "C", "P5_DIR"]:
    assert n in globals(), f"missing {n}: run the earlier cells first"
SECOND_PATTERNS = alternating_patterns(10, [(256, 32), (256, 128)])

def build_second_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True, layer_patterns=SECOND_PATTERNS)
    ck = torch.load(SECOND_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)       # pretrained strided Phase-1 weights
    return RRModel(lm_, 768, 13, dropout).to(device)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0, acting on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_lorma(m, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in m.parameters(): p.requires_grad_(False)
    for layer in m.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, LoRMAPlusLinear(base, r, alpha).to(base.weight.device))
    for p in m.head.parameters(): p.requires_grad_(True)

DEBUG   = False          # True first (~1-2 min); then False
MODE    = "ht4"
N_TRAIN = None
LS = globals().get("LS", {})

random.seed(1); torch.manual_seed(1)
m = ls_from(build_second_rr())
batch = collate_ls("dev", [0, 1], "last1"); m.eval()
with torch.no_grad(): before = m(*batch)
inject_lorma(m, 16, 32)
with torch.no_grad(): d = (before - m(*batch)).abs().max().item()
assert d < 1e-3, f"LoRMA+ is not identity at init ({d})"; print(f"[lorma+] identity at init OK ({d:.1e})")

NAME = f"second_lorma_{MODE}"
LS[NAME] = train_ls(m, NAME, MODE, epochs=2, lr=1e-3, head_lr=1e-3, n_train=N_TRAIN, loss_type="wbce", debug=DEBUG)
if not DEBUG: write_pred_ls(m, NAME, MODE, LS[NAME])

[lorma+] identity at init OK (0.0e+00)
[second_lorma_ht4] mode=ht4 loss=wbce train_docs=42750 steps=9878 trainable=0.81M / 88.3M
  ~28 min training total (excl. dev evals)
  step 200/9878 loss 0.4966
  step 400/9878 loss 0.3664
  step 600/9878 loss 0.4870
  step 800/9878 loss 0.4915
  step 1000/9878 loss 0.1755
  step 1200/9878 loss 0.6833
  step 1400/9878 loss 0.5641
  step 1600/9878 loss 0.5320
  step 1800/9878 loss 0.4957
  step 2000/9878 loss 0.3361
  step 2200/9878 loss 0.4762
  step 2400/9878 loss 0.5089
  eval@2469: macro-F1 @0.5 0.0787 | @best global thr 0.225: 0.1048  (8.7 min)
  step 2600/9878 loss 0.5075
  step 2800/9878 loss 0.5082
  step 3000/9878 loss 0.2442
  step 3200/9878 loss 0.5620
  step 3400/9878 loss 0.8194
  step 3600/9878 loss 0.6880
  step 3800/9878 loss 0.3621
  step 4000/9878 loss 0.6557
  step 4200/9878 loss 0.5457
  step 4400/9878 loss 0.5144
  step 4600/9878 loss 0.4136
  step 4800/9878 loss 0.2083
  eval@4938: macro-F1 @0.5 0.0360 | @best global thr 0.125